# Prob-Edge RND Backtest — Reviewer Notebook

**A proper density extraction beats the broker's ATM expected move on CRPS, out of sample, consistently across every name and regime; the vanilla Prob-Edge cone did not. The one residual is a small 95%-tail gap from wing modeling — disclosed, not a measure effect.**

12 monthly 2025 expiries x SPY/QQQ/AAPL x 2 DTE = 72 triples, 6 methods. Everything below re-derives from the committed, frozen scores; the 72 option chains under `docs/backtest_12mo/chains/` make it reproducible with no network. See `docs/backtest_12mo/BACKTEST_12MO_REPORT.md` for the full write-up.

Two `corrected` variants are shown side by side (no forced winner): **SVI** (best CRPS, no-arb wings, slightly tight) and **quad/flat-wing** (best body calibration, CRPS still beats the broker).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

sc = pd.read_parquet('docs/backtest_12mo/backtest_scores.parquet')
sc['cov68'] = ((sc.S_T >= sc.q16) & (sc.S_T <= sc.q84)).astype(float)
sc['cov95'] = ((sc.S_T >= sc.q2p5) & (sc.S_T <= sc.q97p5)).astype(float)
ORDER = ['atm_iv_normal','atm_iv_lognormal','delta_pop','bl','corrected','corrected_quad']
sc['method'] = pd.Categorical(sc.method, ORDER, ordered=True)
print(len(sc), 'scored rows;', sc.method.nunique(), 'methods;', sc.ticker.nunique(), 'tickers')

## Overall by method (n=72 each; nominal cov 0.68 / 0.95)
CRPS is the headline (lower = better). Winkler95 is the tail-weighted interval score.

In [ ]:
overall = (sc.groupby('method', observed=True)
  .agg(cov68=('cov68','mean'), cov95=('cov95','mean'),
       Winkler68=('winkler68','mean'), Winkler95=('winkler95','mean'),
       CRPS=('crps','mean'), PIT=('pit','mean'), tail_clip=('tail_clip','sum'))
  .round(2))
overall

## CRPS by regime — the durable claim
Both `corrected` variants beat the broker (`atm_iv_normal`) on CRPS in every regime; `bl` never does.

In [ ]:
pd.pivot_table(sc, index='regime', columns='method', values='crps',
               aggfunc='mean', observed=True).round(2)[ORDER]

## Winkler95 by regime — where the residual lives
The broker keeps the tail edge in high vol (corrected wings under-cover at 95% there).

In [ ]:
pd.pivot_table(sc, index='regime', columns='method', values='winkler95',
               aggfunc='mean', observed=True).round(1)[ORDER]

## CRPS by ticker

In [ ]:
pd.pivot_table(sc, index='ticker', columns='method', values='crps',
               aggfunc='mean', observed=True).round(2)[ORDER]

## PIT histograms (flat = calibrated)
U-shape = under-dispersed (too tight); centre-piled = over-dispersed (too wide).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
edges = np.linspace(0, 1, 11)
for ax, m in zip(axes.ravel(), ORDER):
    p = sc.loc[sc.method == m, 'pit'].dropna().to_numpy()
    ax.hist(p, bins=edges, color='#1e50b4', edgecolor='white')
    ax.axhline(len(p) / 10, color='#ff5a82', ls='--', lw=1, label='uniform')
    ax.set_title(m); ax.set_xlim(0, 1); ax.set_xlabel('PIT')
axes[0,0].set_ylabel('count'); axes[1,0].set_ylabel('count')
axes[0,0].legend(fontsize=8); fig.suptitle('PIT by method (n=72 each)')
fig.tight_layout(); plt.show()

## Health & caveats
- IV inversion >=69%/chain; far-OTM no-trade strikes flagged NaN, not guessed.
- `bl` tail-clips 27/72, `delta_pop` 22/72 (window edge) -> their scores are understated; corrected & ATM baselines never clip.
- 0 CRPS-truncated, 0 dropped, 0 producer errors across 432 rows.
- n=72 aggregate robust; per-regime/ticker cells n=24 (directional).
- Guardrails: every density fit only to construction-time option data; no parameter fit to realized S_T; scored strictly out of sample.
- Building stopped at SVI by plan; SSVI / high-vol wing widening could target the residual 95% under-coverage but were not pursued.

In [ ]:
# Accounting from the persisted (already-scored) rows -- no re-derivation needed.
{'n_scored': len(sc),
 'coverage68': round(sc.cov68.mean(), 3), 'coverage95': round(sc.cov95.mean(), 3),
 'crps_mean': round(sc.crps.mean(), 3),
 'n_crps_truncated': int(sc.crps_truncated.sum()),
 'n_tail_clip': int(sc.tail_clip.sum())}